<a href="https://colab.research.google.com/github/scn0901/Intro-to-LLMs-for-Social-Science/blob/main/course_notebooks/day_2_models_to_tools/day2_from_models_to_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2: From Models to Tools

**LLMs for Social Science**

| Module | Topic | Status |
|-----|-------|--------|
| 1 | From Embeddings to Transformers | Done |
| **2** | **From Models to Tools** | **This module** |
| 3 | Deploying for Research | Next |
| 4 | Social Science Applications | |
| 5 | Agentic Workflows | |

## Why this matters

In Module 1, you built a language model's core loop from scratch: tokenize, attend, predict, sample. You saw that a base model can generate fluent text but cannot follow instructions. It just completes whatever text you give it.

In this module we close that gap. You will learn:

1. **What turns a base model into an assistant** (post-training: SFT, RLHF, DPO)
2. **How to use prompts as a research instrument** (zero-shot, few-shot, chain-of-thought)
3. **How to choose the right model** for your research task

By the end of this module, you will have classified real political tweets using multiple prompting strategies, measured how sensitive your results are to prompt wording, and saved your outputs for Module 3's validation pipeline.

## Outline

- **Section 1:** From Base Models to Assistants (~30 min)
- **Section 2:** Prompting as Experimental Design (~75 min)
- **Section 3:** The Model Landscape and Bridge to Module 3 (~25 min)

## Setup

In [1]:
#@title Install dependencies and import libraries
!pip install -q transformers accelerate torch pandas scikit-learn tqdm

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import classification_report
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [2]:


#@title Load dataset: Women's March tweets (Bestvater & Monroe, 2022)

DATA_PATH = 'https://raw.githubusercontent.com/antndlcrx/oss_2024/main/data/WM_tweets_groundtruth.csv'
wm_data = pd.read_csv(DATA_PATH)

# Clean up: create text labels and remove URLs
wm_data['stance_cat'] = wm_data['stance'].map({1: 'support', 0: 'oppose'})
wm_data['sentiment_cat'] = wm_data['sentiment'].map({1.0: 'positive', 0.0: 'negative'})
wm_data['text_cleaned'] = wm_data['text'].str.replace(r'http\S+|www.\S+', '', case=False, regex=True).str.strip()

# Sample a validation set (250 tweets) and a separate pool for few-shot examples
val_data = wm_data.sample(n=250, random_state=42).reset_index(drop=True)
train_pool = wm_data.drop(val_data.index).reset_index(drop=True)

print(f"Validation set: {len(val_data)} tweets")
print(f"Training pool (for few-shot examples): {len(train_pool)} tweets")
print(f"Stance distribution: {val_data['stance_cat'].value_counts().to_dict()}")

Validation set: 250 tweets
Training pool (for few-shot examples): 19362 tweets
Stance distribution: {'support': 211, 'oppose': 39}


In [3]:
wm_data.head(4)

,text,stance,sentiment,balanced_train,vader_scores,stance_cat,sentiment_cat,text_cleaned
0,YES! I'm still with her and always will be. ht...,1,1.0,0.0,0.5754,support,positive,YES! I'm still with her and always will be.
1,Pics or it didn't happen. https://t.co/o1GddSmwk2,1,0.0,0.0,0.0000,support,negative,Pics or it didn't happen.
2,I love this nasty woman. @MaribethMonroe #wome...,1,1.0,1.0,-0.0129,support,positive,I love this nasty woman. @MaribethMonroe #wome...
3,RT @YiawayYeh: Marching for love. Nashville #...,1,1.0,1.0,0.6369,support,positive,RT @YiawayYeh: Marching for love. Nashville #...


---

# Section 1: From Base Models to Assistants

In Module 1, you saw that a base language model (GPT-2) generates fluent text but cannot follow instructions. Ask it to classify a tweet, and it will write another tweet, or continue with a news article, or produce something else entirely. It is not being difficult: it was trained to predict the next token, so that is exactly what it does.

Something happens between "base model" and "ChatGPT" that turns a next-token predictor into something that answers questions, follows instructions, and refuses harmful requests. That something is **post-training**.

## Demo: The gap between base and instruct models

Let's load two versions of the same model family: a **base** model and an **instruction-tuned** model. Same architecture, same size, same pre-training data. The only difference is what happened after pre-training.

In [4]:
#@title Load base and instruct models

BASE_MODEL = "Qwen/Qwen2.5-0.5B"
INSTRUCT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# Base model — a raw next-token predictor
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16
).to(device).eval()

# Instruct model — same architecture, post-trained to follow instructions
inst_tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL)
inst_model = AutoModelForCausalLM.from_pretrained(
    INSTRUCT_MODEL, torch_dtype=torch.float16
).to(device).eval()

print(f"Loaded both models on {device}")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded both models on cuda


In [5]:
#@title Helper: generate from base and instruct models

def generate_base(prompt, max_new_tokens=60):
    """Generate a text continuation from the base model."""
    inputs = base_tokenizer(prompt, return_tensors="pt").to(device)
    out = base_model.generate(inputs["input_ids"], max_new_tokens=max_new_tokens, do_sample=False)
    return base_tokenizer.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=True)

def generate_instruct(prompt, max_new_tokens=60):
    """Generate a response from the instruct model using its chat format."""
    messages = [{"role": "user", "content": prompt}]
    text = inst_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = inst_tokenizer([text], return_tensors="pt").to(device)
    out = inst_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[0][len(inputs["input_ids"][0]):]
    return inst_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

### Three tasks that reveal the gap

Write your own prompt for each task below and run the cells one at a time. The same prompt goes to both the base model and the instruct model; pay attention to what each one does with it.

**Task 1: Letter counting**

Write a prompt asking the model how many times the letter *r* appears in the word *strawberry*. Tell it to reply with a number only.

In [6]:
task1_prompt = "How many times the letter r appears in the word strawberry? Please reply with a number only."  # YOUR CODE HERE

if not task1_prompt:
    print("Write your prompt above, then re-run.")
else:
    print("BASE MODEL  →", generate_base(task1_prompt, max_new_tokens=50))
    print("INSTRUCT    →", generate_instruct(task1_prompt, max_new_tokens=50))

BASE MODEL  →  To determine how many times the letter 'r' appears in the word "strawberry," we need to count the occurrences of the letter 'r' in the given word. Let's break it down step by step:

1. **Identify
INSTRUCT    → 1


**Task 2: Word reversal**

Write a prompt asking the model to write the word *python* backwards. Tell it to give only the reversed word, nothing else.

In [7]:
task2_prompt = "Write the word python backwards. Please give only the reversed word, nothing else."  # YOUR CODE HERE

if not task2_prompt:
    print("Write your prompt above, then re-run.")
else:
    print("BASE MODEL  →", generate_base(task2_prompt, max_new_tokens=50))
    print("INSTRUCT    →", generate_instruct(task2_prompt, max_new_tokens=50))

BASE MODEL  →  The word python backwards is "taypoe".
INSTRUCT    → nohtyp.


**Task 3: Roleplay**

Write a prompt that sets up a scenario: the model is an excited medieval blacksmith who has just seen a microwave oven. Ask it to describe the object in one sentence.

In [8]:
task3_prompt = "You are an excited medieval blacksmith who has just seen a microwave oven. Please describe the object in one sentence."  # YOUR CODE HERE

if not task3_prompt:
    print("Write your prompt above, then re-run.")
else:
    print("BASE MODEL  →", generate_base(task3_prompt, max_new_tokens=80))
    print("INSTRUCT    →", generate_instruct(task3_prompt, max_new_tokens=80))

BASE MODEL  →  The microwave oven is a marvel of modern technology, a marvel of modern technology, a marvel of modern technology, a marvel of modern technology, a marvel of modern technology, a marvel of modern technology, a marvel of modern technology, a marvel of modern technology, a marvel of modern technology, a marvel of modern technology, a marvel of modern technology, a marvel of modern technology, a marvel of modern
INSTRUCT    → A medieval blacksmith's eyes widen as they behold the sleek and modern marvel of a microwave oven, its metallic casing gleaming under the sunlight, and the warmth radiating from its powerful heating element that seems to pulse with life.


The pattern across all three tasks:

- **Base model** treats the prompt as text to continue. It predicts plausible next tokens given the full prompt as context, but it does not recognise the instruction *as an instruction*. You are providing it with a longer prefix to extend.
- **Instruct model** recognises the prompt as a request and attempts to fulfil it. It may or may not succeed — but it is clearly *trying to answer*, not generating more text.

**Task 1: why letter counting is hard even after instruction tuning:** recall from Module 1 that models tokenise text into sub-word pieces. "Strawberry" is typically split into something like `straw` + `berry`; individual letters are not directly represented. The instruct model understands it should count 'r's, but the actual counting requires attending to character-level information that is hidden inside tokens. This is a capability limitation, not a post-training failure: SFT teaches the model *to try*, it cannot teach the model an ability it does not have.

**Things to notice:**
- Did you need to word the prompt in any particular way to get the instruct model to answer with just a number or a single word?
- Did the base model ever accidentally follow your instruction, or did it always drift into continuation?
- Did the instruct model ever get the letter count right? If not, why?

### Generation parameters playground

So far both helper functions use `do_sample=False` (greedy decoding): at every step the model picks the single most probable next token. The output is deterministic — run it twice and you get the same result — but it can be overly conservative or repetitive.

Real inference pipelines expose several parameters to trade off diversity against coherence:

| Parameter | What it does | Effect |
|-----------|-------------|--------|
| `do_sample` | Switch between greedy (`False`) and sampling (`True`) | Greedy is deterministic; sampling is stochastic |
| `temperature` | Scales logits before sampling. Only active when `do_sample=True`. | Low (0.1) → peaked, repetitive. High (2.0) → flat, incoherent |
| `top_k` | Only sample from the *k* most probable tokens. `0` = disabled. | Truncates the long tail of unlikely tokens |
| `top_p` | Only sample from the smallest set of tokens whose cumulative probability ≥ *p* (nucleus sampling). `1.0` = disabled. | Adapts the effective vocabulary size to the local distribution |

The cell below runs the instruct model with whatever parameters you set. Change the values and re-run to build intuition. Switch the `task` variable to try any of the three prompts from above.

In [9]:
#@title Generation parameters playground — edit the values and re-run

# ── Which prompt to use? ──────────────────────────────────────────────────────
# Use one of your prompts from above, or write a new one here.
task = task3_prompt  # or task2_prompt, task3_prompt, or any string

# ── Generate function ────────────────────────────────────────────────────────────────
def generate_instruct(prompt, max_new_tokens=60):
    """Generate a response from the instruct model using its chat format."""
    messages = [{"role": "user", "content": prompt}]
    text = inst_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = inst_tokenizer([text], return_tensors="pt").to(device)

    out = inst_model.generate(**inputs,
    # CHANGE GENERATION PARAMETERS HERE!
     max_new_tokens=max_new_tokens,
      do_sample=True,
      temperature=0.1,
      top_k=0.0,
      top_p=1.0)
    new_tokens = out[0][len(inputs["input_ids"][0]):]
    return inst_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print(generate_instruct(task, max_new_tokens=80))

A medieval blacksmith marvels at the sleek and efficient microwave oven, its compact design and powerful heating capabilities setting new standards for modern household appliances.


### Now with a real research task

The same gap appears when the task is actually useful: classifying a political tweet.
Run the cell below and compare the two outputs.

In [10]:
example_tweet = val_data['text_cleaned'].iloc[0]
instruction = (
    f"Classify this tweet as supporting or opposing the Women's March.\n"
    f"Answer with one word: support or oppose.\n\n"
    f"Tweet: {example_tweet}\n"
    f"Answer: "
)

print("BASE MODEL:")
print(generate_base(instruction)[:300])

print("\nINSTRUCT MODEL:")
print(generate_instruct(instruction)[:300])

BASE MODEL:
1. Support

The tweet is expressing support for the Women's March, as it mentions that the protesters left their "after their #WomensMarch" which implies they were not against the march. This indicates that the protesters were not against the march and were simply leaving after it. Therefore, the

INSTRUCT MODEL:
support


The difference is stark — and it mirrors what you saw in the toy tasks above. The base model treats your instruction as text to continue. The instruct model treats it as a request to fulfil.

Both models have identical "knowledge": same architecture, same pre-training data, same weights at the start of post-training. The only difference is what happened after pre-training. When a model fails at your research task, that failure is almost never a post-training problem. It is either a capability limitation from pre-training (like the letter-counting task) or a prompting problem — which is what the rest of this module addresses.

## The post-training pipeline

Three stages turn a base model into an assistant:

**1. Supervised Fine-Tuning (SFT)**

The model is shown thousands of (instruction, good response) pairs and trained to imitate the good responses. This is like giving someone a style guide with worked examples: "When asked X, respond like Y."

After SFT, the model can follow instructions. But it has no sense of what makes one valid response *better* than another.

**2. Preference Optimization (RLHF or DPO)**

Human annotators compare pairs of model outputs and choose which is better. This preference data teaches the model to favor responses that are more helpful, more accurate, and safer.

- **RLHF** (Reinforcement Learning from Human Feedback): trains a separate "reward model" on the preference data, then uses reinforcement learning to optimize the main model against that reward.
- **DPO** (Direct Preference Optimization): skips the reward model and optimizes preferences directly. Simpler, often comparably effective.

**3. Safety training**

Additional fine-tuning to reduce harmful outputs, using techniques like Constitutional AI (Anthropic) or red-teaming. The model learns to refuse dangerous requests, flag uncertainty, and avoid generating toxic content.

The key intuition: SFT teaches *format* (how to respond). Preference optimization teaches *quality* (which responses are better). Safety training teaches *boundaries* (when not to respond).

## What post-training does NOT change

Post-training reshapes behavior, but it does not add new knowledge. The model's factual knowledge, language capabilities, and reasoning ability all come from pre-training. If the base model does not know a fact, the instruct model does not know it either. It will just express its ignorance more politely (or, worse, make up an answer more convincingly).

This distinction matters for research: when a model gets a classification wrong, it is usually not a post-training problem. It is a capability limitation from pre-training, or a prompting problem. Post-training just determines whether the model gives you a clean one-word answer or a rambling paragraph.

**Section takeaway: Post-training turns a text-completion engine into a useful tool. SFT teaches format, preference optimization teaches quality, and safety training teaches boundaries. But post-training is also a set of design choices about what "good" means, and those choices affect your research outputs.**

In [11]:
#@title Free GPU memory: remove the base model (we only need instruct from here)
import gc
del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Base model removed. Instruct model still loaded.")

Base model removed. Instruct model still loaded.


## A note on the pipeline API

We loaded our models manually using `AutoModelForCausalLM`. For the classification exercises that follow, we will use Hugging Face's `pipeline` API, which wraps tokenization, inference, and decoding into a single callable:

```python
from transformers import pipeline
pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

# For instruct models, pass a list of message dicts:
messages = [{"role": "user", "content": "Classify this tweet..."}]
output = pipe(messages, max_new_tokens=30)
response = output[0]["generated_text"][-1]["content"]
```

The pipeline handles chat templates, GPU placement, and decoding for you. In Module 1 you built all of this from scratch; now you can treat the model as a function from text to text.

When working at scale (Module 3), you will switch to API calls to commercial models. The pattern is the same: send a message, get a response.

# Section 2: Prompting as Experimental Design

For social scientists, a prompt is not just a way to talk to a model. It is a **measurement instrument**. The wording of your prompt determines what construct you measure, how reliably you measure it, and whether your results replicate.

In this section, you will classify real political tweets using progressively more sophisticated prompting strategies. Along the way, you will discover that prompt design requires the same rigor as survey design: small wording changes can produce large differences in results.

## The dataset

We are working with the Women's March Twitter dataset from [Bestvater and Monroe (2022)](https://www.cambridge.org/core/services/aop-cambridge-core/content/view/743A9DD62DF3F2F448E199BDD1C37C8D/S1047198722000109a.pdf). The dataset contains tweets about the 2017 Women's March, labeled for both **stance** (support vs. oppose the march) and **sentiment** (positive vs. negative tone).

The authors' key finding: sentiment is not stance. A tweet can oppose the march in a positive tone, or support it in a negative tone. We will return to this distinction later.

In [12]:
# Sample tweets from each class
for label in ['support', 'oppose']:
    print(f"{label.upper()}:")
    for _, row in val_data[val_data['stance_cat'] == label].head(3).iterrows():
        print(f"  {row['text_cleaned'][:120]}")
    print()

SUPPORT:
  This is what they left after their #WomensMarch
  I work here #Oakland #notmypresident #womensmarch #resistance #bayarea @ Frank Ogawa Plaza
  RT @LauraHuu: We want free birth control in order to avoid making mistakes like you. #WomensMarch

OPPOSE:
  RT @bocavista2016: If the organizer of #WomensMarch wore a vagina on her head

She'd be EXECUTED as per Sharia


#F
  LOL...shut up and sit down you old fool

#WomensMarch



In [13]:
#@title Classification helpers

# --- Pipeline setup ---
inst_tokenizer.padding_side = 'left'  # required for batch generation

instruct_pipe = pipeline(
    "text-generation",
    model=inst_model,
    tokenizer=inst_tokenizer,
    device=device,
)

# --- Helper functions ---

def extract_label(response_text, labels):
    """Find the first matching label in the model's response."""
    resp = response_text.lower().strip()
    for label in labels:
        if label in resp:
            return label
    return "unknown"


def classify_batch(pipe, user_messages, max_new_tokens=30):
    """Run batch inference on a list of user messages. Returns raw response strings."""
    prompts = [
        pipe.tokenizer.apply_chat_template(
            [{"role": "user", "content": msg}],
            tokenize=False, add_generation_prompt=True
        )
        for msg in user_messages
    ]
    outputs = pipe(
        prompts,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        batch_size=len(prompts),
    )
    return [out[0]['generated_text'][len(p):].strip() for out, p in zip(outputs, prompts)]


def run_classification(val_data, prompt_template, labels=("support", "oppose"), batch_size=8):
    """Classify all tweets in val_data using the given prompt template.
    Returns a list of predicted labels.
    """
    messages = [prompt_template.format(text=row['text_cleaned']) for _, row in val_data.iterrows()]
    all_preds = []
    for i in tqdm(range(0, len(messages), batch_size), desc="Classifying"):
        batch = messages[i:i+batch_size]
        responses = classify_batch(instruct_pipe, batch)
        all_preds.extend([extract_label(r, labels) for r in responses])
    return all_preds

print("Helpers ready.")

Helpers ready.


### Exercise 2: Zero-shot Classification

Write a prompt that asks the model to classify each tweet as "support" or "oppose." No examples, no elaborate instructions: just a clear request.

The prompt should be a string with `{text}` where the tweet goes. For example:
```
"Classify this text as positive or negative: {text}\nAnswer:"
```
(But write your own, tailored to our task.)

**Hint:** Think about what information the model needs: the task (classify), the options (support/oppose), and where the tweet is.

In [32]:
# Write your prompt template. Use {text} as the placeholder for the tweet.
# Note: Use a regular string (not an f-string) so {text} stays as a placeholder.
prompt_zero_shot = "Does the author of the tweet support or oppose the Women's March? Answer with one word: 'support' or 'oppose'.\n\nTweet: {text}\nAnswer:"  # YOUR CODE HERE

# Run classification
if prompt_zero_shot:
    val_data['pred_zero'] = run_classification(val_data, prompt_zero_shot)
    print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))

Classifying: 100%|██████████| 32/32 [00:03<00:00,  8.39it/s]

              precision    recall  f1-score   support

      oppose      0.173     1.000     0.294        39
     support      1.000     0.114     0.204       211

    accuracy                          0.252       250
   macro avg      0.586     0.557     0.249       250
weighted avg      0.871     0.252     0.218       250



### Exercise 3: Few-shot Classification

Zero-shot gave us a baseline. Now let's see if providing examples helps. In **few-shot prompting**, we include labeled examples in the prompt so the model can learn the pattern before classifying the target tweet.

The helper below builds a few-shot prompt by sampling balanced examples from the training pool. Study it, then run the classification with different values of `n_per_class` (1, 2, 3, 5). Is there a point of diminishing returns?

**Note:** With a 0.5B model, few-shot gains may be modest. With frontier models (GPT-4o, Claude), the gains are typically larger. If you want, try changing the model at the top of the notebook to `Qwen/Qwen2.5-3B-Instruct` and re-running.

In [33]:
#@title Few-shot prompt builder

def build_few_shot_prompt(text, train_df, n_per_class=2, seed=42):
    """
    Build a few-shot prompt with n_per_class examples from each class.
    Examples are shuffled so the model does not always see one class first.
    """
    support_ex = train_df[train_df['stance_cat'] == 'support'].sample(n=n_per_class, random_state=seed)
    oppose_ex  = train_df[train_df['stance_cat'] == 'oppose'].sample(n=n_per_class, random_state=seed)
    examples = pd.concat([support_ex, oppose_ex]).sample(frac=1, random_state=seed)

    demo_lines = [
        f"Tweet: {row['text_cleaned']}\nAnswer: {row['stance_cat']}"
        for _, row in examples.iterrows()
    ]

    return (
        "Does the author of each tweet support or oppose the Women's March? "
        "Answer with one word: support or oppose.\n\n"
        + "\n\n".join(demo_lines)
        + f"\n\nTweet: {text}\nAnswer:"
    )

# Preview
print(build_few_shot_prompt(val_data['text_cleaned'].iloc[0], train_pool, n_per_class=2))

Does the author of each tweet support or oppose the Women's March? Answer with one word: support or oppose.

Tweet: RT @erinLOLiver: .@HillaryClinton @womensmarch Grace Under Fire. You have opened doors  speaking Truth To Power. It was my honor to campaig…
Answer: support

Tweet: RT @B_ails: We literally have the same rights as men. You can't get more equal than that. #WomensMarch
Answer: oppose

Tweet: RT @LorenaSGonzalez: Today is the most hopeful I have felt since the election! #WomensMarch #sandiegowomensmarch
Answer: support

Tweet: #WomensMarch governors/mayors verify cop voter registration;send only dem cops to dem cities;send gop cops to gop cities for gop problems
Answer: oppose

Tweet: This is what they left after their #WomensMarch
Answer:


In [34]:
# Try changing n_per_class: 1, 2, 3, 5
n_per_class = 5

# Build few-shot prompts for every tweet
few_shot_prompts = [
    build_few_shot_prompt(row['text_cleaned'], train_pool, n_per_class=n_per_class)
    for _, row in val_data.iterrows()
]

# Run inference (batch_size=1 because few-shot prompts are long)
preds_few = []
for i in tqdm(range(len(few_shot_prompts)), desc="Few-shot"):
    responses = classify_batch(instruct_pipe, [few_shot_prompts[i]], max_new_tokens=10)
    preds_few.append(extract_label(responses[0], ("support", "oppose")))

val_data['pred_few'] = preds_few

print("\nZERO-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))
print("FEW-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_few'], digits=3))

Few-shot:   1%|          | 2/250 [00:00<00:25,  9.58it/s][transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Few-shot:   2%|▏         | 4/250 [00:00<00:22, 10.72it/s][transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_


ZERO-SHOT:
              precision    recall  f1-score   support

      oppose      0.173     1.000     0.294        39
     support      1.000     0.114     0.204       211

    accuracy                          0.252       250
   macro avg      0.586     0.557     0.249       250
weighted avg      0.871     0.252     0.218       250

FEW-SHOT:
              precision    recall  f1-score   support

      oppose      0.000     0.000     0.000        39
     support      0.843     0.995     0.913       211

    accuracy                          0.840       250
   macro avg      0.422     0.498     0.457       250
weighted avg      0.712     0.840     0.771       250



### Exercise 4: Prompt Sensitivity

This is the most important exercise for your future research.

Small changes to prompt wording can produce surprisingly large differences in classification results. If your prompt is your measurement instrument, you need to know how stable it is.

Create **three variations** of the zero-shot classification prompt. Change only the wording, not the fundamental task:

- **Variation 1:** Rephrase the instruction ("Is this tweet in favor of or against..." vs. "Does the author support or oppose...")
- **Variation 2:** Change the label names ("support/oppose" vs. "pro/anti" vs. "favorable/unfavorable")
- **Variation 3:** Change the structure (put the tweet before the instruction instead of after)

Run all three on the same 250 tweets. Compare the classification reports.

**Important:** If you change the label names (e.g. to "pro"/"anti"), you need to tell the code what labels to look for and how they map back to the ground truth ("support"/"oppose"). The cell below handles this for you — just fill in the `labels` and `label_map` for each variation.

In [35]:
# Each variation needs three things:
#   prompt  — the prompt template with {text}
#   labels  — what words to look for in the model's response
#   label_map — how to convert those words back to "support"/"oppose" for evaluation
#
# If your labels are already "support"/"oppose", the map is trivial.
# If you use "pro"/"anti", the map converts them back.

variations = {
    "v1_rephrase": {
        "prompt": "Is this tweet in favor of or against the Women's March? Answer with one word: 'support' or 'oppose'.\n\nTweet: {text}\nAnswer:",  # YOUR CODE HERE: rephrase the instruction
        "labels": ("support", "oppose"),
        "label_map": {"support": "support", "oppose": "oppose"},
    },
    "v2_relabel": {
        "prompt": "Does the author of the tweet support or oppose the Women's March? Answer with one word: 'pro' or 'anti'.\n\nTweet: {text}\nAnswer:",  # YOUR CODE HERE: change the label names (e.g. pro/anti)
        "labels": ("pro", "anti"),           # <-- change these to match your prompt
        "label_map": {"pro": "support", "anti": "oppose"},  # <-- map back to ground truth
    },
    "v3_reorder": {
        "prompt": "Tweet: {text}\n\nDoes the author of the tweet support or oppose the Women's March? Answer with one word: 'support' or 'oppose'.\nAnswer:",  # YOUR CODE HERE: change the structure
        "labels": ("support", "oppose"),
        "label_map": {"support": "support", "oppose": "oppose"},
    },
}

In [36]:
# Run all variations and evaluate
for name, cfg in variations.items():
    if not cfg["prompt"]:
        print(f"Skipping {name}: no prompt defined\n")
        continue

    # Classify
    raw_preds = run_classification(val_data, cfg["prompt"], labels=cfg["labels"])

    # Map predictions back to ground-truth label space
    mapped_preds = [cfg["label_map"].get(p, "unknown") for p in raw_preds]
    val_data[f'pred_{name}'] = mapped_preds

    print(f"\n{name.upper()}:")
    print(classification_report(val_data['stance_cat'], mapped_preds, digits=3))

Classifying:  25%|██▌       | 8/32 [00:00<00:02,  8.91it/s][transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Classifying:  75%|███████▌  | 24/32 [00:02<00:00,  9.10it/s][transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max


V1_REPHRASE:
              precision    recall  f1-score   support

      oppose      0.453     0.744     0.563        39
     support      0.956     0.825     0.885       211
     unknown      0.000     0.000     0.000         0

    accuracy                          0.812       250
   macro avg      0.470     0.523     0.483       250
weighted avg      0.878     0.812     0.835       250



Classifying:   0%|          | 0/32 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Classifying:   6%|▋         | 2/32 [00:00<00:02, 12.90it/s][transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_toke


V2_RELABEL:
              precision    recall  f1-score   support

      oppose      0.182     1.000     0.308        39
     support      1.000     0.171     0.291       211

    accuracy                          0.300       250
   macro avg      0.591     0.585     0.300       250
weighted avg      0.872     0.300     0.294       250



Classifying: 100%|██████████| 32/32 [00:03<00:00,  8.02it/s]



V3_REORDER:
              precision    recall  f1-score   support

      oppose      0.221     0.974     0.360        39
     support      0.987     0.365     0.533       211

    accuracy                          0.460       250
   macro avg      0.604     0.670     0.447       250
weighted avg      0.868     0.460     0.506       250



If your results depend on how you phrase the prompt, you have a reproducibility problem. This is the prompting equivalent of question wording effects in survey design.

### Exercise 5: Chain-of-Thought

For ambiguous tweets, asking the model to *explain its reasoning* before giving a label can improve accuracy — this is **chain-of-thought (CoT) prompting**.

Write a CoT prompt that first asks the model to explain the author's position in one sentence, then classify as support or oppose. Run it on all 250 tweets so you can compare the classification report directly with zero-shot and few-shot.

**Hint:** Structure your prompt so the final label appears on its own clearly-delimited line — for example, ending with `"Classification: support"` or `"Classification: oppose"`. This makes extraction reliable.

**Note on cost:** CoT responses are much longer than single-word answers — you are asking for 80–120 tokens per tweet instead of 1–2. At inference time this is slower; with API-based models it means more money per classification. Whether the accuracy gain justifies that cost depends on your task and budget.

In [ ]:
# Write a CoT prompt. Use {text} as the placeholder for the tweet.
prompt_cot = ""  # YOUR CODE HERE

def extract_label_cot(response_text, labels=("support", "oppose")):
    """Check lines from the end so reasoning text doesn't pollute the label."""
    lines = response_text.lower().strip().split("\n")
    for line in reversed(lines):
        for label in labels:
            if label in line:
                return label
    return "unknown"

if prompt_cot:
    messages = [prompt_cot.format(text=row['text_cleaned']) for _, row in val_data.iterrows()]
    cot_preds = []
    for i in tqdm(range(0, len(messages), 4), desc="CoT"):
        responses = classify_batch(instruct_pipe, messages[i:i+4], max_new_tokens=120)
        cot_preds.extend([extract_label_cot(r) for r in responses])
    val_data['pred_cot'] = cot_preds
    print(classification_report(val_data['stance_cat'], val_data['pred_cot'], digits=3))

CoT gives the model a chance to work through ambiguous cases before committing to a label. Whether it actually helps depends on your prompt and the model's reasoning capacity — a 0.5B model has limited ability to reason, so gains may be modest or absent. With frontier reasoning models (o1, DeepSeek-R1, Claude with extended thinking) the gains are typically larger because reasoning is baked into the model itself.

The cost tradeoff is also real beyond just time: if CoT gives +2 F1 points but quadruples your token usage across thousands of documents, you need to decide whether that gain is worth it for your specific research question.

### Exercise 6: Stance vs. Sentiment

The Bestvater and Monroe paper makes a point that matters for every social scientist using LLMs: **what you ask for is what you get.** If you ask for the wrong construct, you measure the wrong thing.

Change your prompt from classifying **stance** (support/oppose) to classifying **sentiment** (positive/negative). Run it on the same tweets. Then compare the sentiment predictions against the true *stance* labels.

If stance and sentiment were the same construct, the results would match. They won't.

In [ ]:
prompt_sentiment = (
    "What is the sentiment of the following tweet? "
    "Answer with one word: positive or negative.\n\n"
    "Tweet: {text}\n"
    "Answer:"
)

preds_sentiment = run_classification(val_data, prompt_sentiment, labels=("positive", "negative"))

# Map sentiment to stance labels for comparison
preds_mapped = [
    "support" if p == "positive" else "oppose" if p == "negative" else "unknown"
    for p in preds_sentiment
]

print("SENTIMENT predictions vs. true STANCE labels:")
print(classification_report(val_data['stance_cat'], preds_mapped, digits=3))
print("\nCompare to our zero-shot STANCE classification:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))

Sentiment and stance are different constructs. A tweet can support the Women's March in an angry tone (positive stance, negative sentiment) or oppose it politely (negative stance, positive sentiment). Your prompt defines your construct. Get it wrong, and you measure the wrong thing.

**Section takeaway: Prompting is experimental design.** Your prompt wording determines what you measure (construct validity), how reliably you measure it (reproducibility), and whether additional complexity (few-shot, CoT) is worth the cost. Treat prompt design with the same rigor you would apply to survey question design.

In [ ]:
#@title Results summary: all methods compared

from sklearn.metrics import f1_score, accuracy_score

methods = {}
for col, label in [("pred_zero", "Zero-shot"),
                   ("pred_few",  "Few-shot"),
                   ("pred_cot",  "Chain-of-thought")]:
    if col not in val_data.columns:
        continue
    known = val_data[col] != "unknown"
    methods[label] = {
        "F1 (macro)": f1_score(val_data.loc[known, 'stance_cat'], val_data.loc[known, col], average='macro'),
        "Accuracy":   accuracy_score(val_data.loc[known, 'stance_cat'], val_data.loc[known, col]),
        "% unknown":  round(100 * (~known).mean(), 1),
    }

if methods:
    print(pd.DataFrame(methods).T.sort_values("F1 (macro)", ascending=False).to_string(float_format="{:.3f}".format))
else:
    print("Run Exercises 2, 3, and 5 first to populate this table.")

---

# Section 3: The Model Landscape and Bridge to Module 3

You have been working with a 0.5-billion-parameter open model. There are hundreds of models available, ranging from 0.5B to over 400B parameters, open and closed. How do you choose?

## Benchmarks: what they measure and what they miss

The most common model benchmarks:

| Benchmark | What it measures | Limitation |
|-----------|-----------------|------------|
| **MMLU** | Broad knowledge across 57 subjects | Models may have memorized test questions |
| **HumanEval** | Code generation ability | Only measures Python, narrow task |
| **GPQA** | Expert-level reasoning (graduate-level science) | Very small test set |
| **Chatbot Arena** | Human preference in open-ended conversation | Reflects general users, not researchers |

**Key limitations of all benchmarks:**

- **Contamination:** models may have seen benchmark questions during training, inflating scores.
- **Gaming:** providers can optimize specifically for benchmarks without improving general ability.
- **Construct validity:** a high MMLU score does not mean the model will classify your political texts well. No benchmark measures *your* task. That is why building task-specific evaluations (which you will do in Module 3) matters more than any leaderboard.

The most ecologically valid benchmark is arguably [Chatbot Arena](https://huggingface.co/spaces/lmarena-ai/chatbot-arena), which ranks models based on blind human preference comparisons on real conversations.

## Open vs. closed models

| | Open models | Closed models |
|---|---|---|
| **Examples** | Llama, Qwen, Mistral, Gemma | GPT-4o, Claude, Gemini |
| **Capability** | Catching up; frontier open models now rival closed | Still ahead on hardest tasks |
| **Cost** | Free to run locally; pay for compute | Pay per token |
| **Transparency** | Full access to weights, can inspect and modify | Black box |
| **Data privacy** | Your data never leaves your machine | Data sent to provider's API |
| **Reproducibility** | Weights are fixed; same model forever | Provider can update silently |

**For social science research:**

- If your data is sensitive (survey responses, medical records, classified documents), open models let you process everything locally.
- If you need maximum capability and your data is not sensitive, closed models are currently ahead.
- If reproducibility matters (it should), open models are safer: a closed model can change between when you run your study and when reviewers try to replicate it.
- The gap between open and closed models is narrowing rapidly.

## What you just experienced

You worked with a 0.5B-parameter open model. It classified political tweets reasonably well but struggled with some ambiguous cases and formatting. A frontier model (100-1000x larger, with extensive post-training) would likely perform better on all of these tasks. But it costs money per token, you cannot run it locally, and you cannot inspect its internals. That tradeoff is real, and there is no universal right answer.

### Exercise 7: Save Your Work (Bridge to Module 3)

Pick your best prompt from Section 2 (whichever gave the best results). Save three things:

1. The prompt template you used
2. The model's predictions on all 250 tweets
3. Your own manual labels for 10 tweets (you will label them yourself right now)

Module 3 opens by loading this file and computing inter-annotator agreement between you and the model.

In [ ]:
# 1. Pick your best prompt
best_prompt = ""  # YOUR CODE HERE

# 2. Pick the best prediction column
best_pred_column = "pred_zero"  # change to "pred_few", "pred_v1_rephrase", etc.

# 3. Manually label 10 tweets. Read each one and decide: support or oppose.
sample_for_labeling = val_data.head(10)
for i, (_, row) in enumerate(sample_for_labeling.iterrows()):
    print(f"[{i}] {row['text_cleaned'][:150]}")
    print(f"    Model said: {row[best_pred_column]}")
    print()

my_labels = [
    "",  # tweet 0
    "",  # tweet 1
    "",  # tweet 2
    "",  # tweet 3
    "",  # tweet 4
    "",  # tweet 5
    "",  # tweet 6
    "",  # tweet 7
    "",  # tweet 8
    "",  # tweet 9
]

In [ ]:
#@title Save results to CSV
output = val_data[['text_cleaned', 'stance_cat', 'sentiment_cat']].copy()
output['model_prediction'] = val_data[best_pred_column]
output['prompt_used'] = best_prompt

output['human_label'] = None
for i, label in enumerate(my_labels):
    if label:
        output.loc[output.index[i], 'human_label'] = label

output.to_csv('day2_classification_results.csv', index=False)
print(f"Saved to day2_classification_results.csv ({len(output)} rows, "
      f"{output['human_label'].notna().sum()} with human labels)")

---

## What you learned in this module

1. **Post-training** turns base models into useful tools. SFT teaches format, preference optimization teaches quality, safety training teaches boundaries. These are design choices that affect your research.

2. **Prompting is experimental design.** Your prompt wording determines what you measure and how reliably. You saw classification accuracy swing with minor rephrasing.

3. **Progressive techniques** (few-shot, CoT) each have their place. Few-shot helps when the model needs calibration. CoT helps with ambiguous cases.

4. **Construct validity matters.** Sentiment is not stance. Your prompt defines your construct.

5. **Model choice** involves real tradeoffs between capability, cost, transparency, and reproducibility.

| Module | Topic | Status |
|-----|-------|--------|
| 1 | From Embeddings to Transformers | Done |
| 2 | From Models to Tools | Done |
| **3** | **Deploying for Research** | **Next** |
| 4 | Social Science Applications | |
| 5 | Agentic Workflows | |

## Bridge to Module 3

You now know how to prompt models and you have seen how sensitive the results are to design choices. Module 3 takes the next step: how to build rigorous classification pipelines at scale (APIs, batching, cost management), how to validate your outputs (inter-annotator agreement, gold-standard sets), and when prompting is not enough and you need to fine-tune.

You will start Module 3 by loading the CSV you just saved and computing agreement metrics.

---
# Extension Exercises
*For students who finish early or want to go deeper.*

---

### Extension A: Structured Output (JSON)

So far, we have been asking the model for a single word. In practice, you often want more structure: a label, a confidence level, maybe a brief justification. Getting a model to return valid JSON makes it much easier to parse outputs in a pipeline.

Write a prompt that requests a JSON response with two fields: `label` (support or oppose) and `confidence` (high, medium, or low). Run on 30 tweets and check format compliance.

In [ ]:
# Write a prompt that asks for JSON output
prompt_json = ""  # YOUR CODE HERE

# Run on a small subset
subset = val_data.head(30)

if prompt_json:
    import json as json_lib
    messages = [prompt_json.format(text=row['text_cleaned']) for _, row in subset.iterrows()]

    json_responses = []
    for i in range(0, len(messages), 4):
        batch = messages[i:i+4]
        json_responses.extend(classify_batch(instruct_pipe, batch, max_new_tokens=80))

    valid = 0
    for i, resp in enumerate(json_responses):
        cleaned = resp.strip().strip('`').strip()
        if cleaned.startswith('json'):
            cleaned = cleaned[4:].strip()
        try:
            parsed = json_lib.loads(cleaned)
            if 'label' in parsed and 'confidence' in parsed:
                valid += 1
                if i < 5:
                    print(f"  [OK] {parsed}")
        except json_lib.JSONDecodeError:
            if i < 5:
                print(f"  [FAIL] {resp[:100]}")

    print(f"\nValid JSON: {valid} / {len(json_responses)} ({100*valid/len(json_responses):.0f}%)")

---
# Solutions

---

In [ ]:
#@title Solution: Exercise 1 (Be the Reward Model)

# Pair 1: B is clearly worse (vague, uninformative). A gives a concrete, structured answer.
#
# Pair 2: B is better. A is confident but one-sided, cherry-picking evidence.
#   B acknowledges complexity and distinguishes aggregate from distributional effects.
#
# Pair 3: B is better. A helps with a request that violates privacy and ToS.
#   This is a safety case. The "helpful" response (A) is actually harmful.
#
# Pair 4: B is better. A overstates the findings ("major breakthrough", "every metric").
#   B reports the results accurately, including limitations.
#
# Pair 5: Genuine judgment call. A is more comprehensive but less actionable.
#   B gives a concrete recommendation but is more opinionated.
#   Different annotators will disagree here, and that disagreement is a real
#   problem for RLHF: the reward model learns from majority preference,
#   which may not match YOUR preferences.

print("See comments above for discussion.")
print("The key insight: you just did what a reward model does.")
print("RLHF automates this at scale. The disagreements you had")
print("(especially on Pair 5) are exactly why alignment is hard.")

In [ ]:
#@title Solution: Exercise 2 (Zero-shot Classification)
prompt_zero_shot = (
    "Does the author of the following tweet support or oppose the Women's March? "
    "Answer with one word: support or oppose.\n\n"
    "Tweet: {text}\n"
    "Answer:"
)

val_data['pred_zero'] = run_classification(val_data, prompt_zero_shot)
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))

In [ ]:
#@title Solution: Exercise 3 (Few-shot Classification)
print("ZERO-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))
print("FEW-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_few'], digits=3))

# With a 0.5B model, few-shot gains may be modest.
# With frontier models (GPT-4o, Claude, Llama-70B), gains are typically larger.
# Try changing the model to Qwen/Qwen2.5-3B-Instruct and re-running!

In [ ]:
#@title Solution: Exercise 4 (Prompt Sensitivity)
variations = {
    "v1_rephrase": {
        "prompt": (
            "Is the following tweet in favor of or against the Women's March? "
            "Answer with one word: support or oppose.\n\n"
            "Tweet: {text}\n"
            "Answer:"
        ),
        "labels": ("support", "oppose"),
        "label_map": {"support": "support", "oppose": "oppose"},
    },
    "v2_relabel": {
        "prompt": (
            "Does the author of the following tweet support or oppose the Women's March? "
            "Answer with one word: pro or anti.\n\n"
            "Tweet: {text}\n"
            "Answer:"
        ),
        "labels": ("pro", "anti"),
        "label_map": {"pro": "support", "anti": "oppose"},
    },
    "v3_reorder": {
        "prompt": (
            "Tweet: {text}\n\n"
            "Based on the tweet above, does the author support or oppose the Women's March? "
            "Answer with one word: support or oppose."
        ),
        "labels": ("support", "oppose"),
        "label_map": {"support": "support", "oppose": "oppose"},
    },
}

print("ORIGINAL ZERO-SHOT:")
print(classification_report(val_data['stance_cat'], val_data['pred_zero'], digits=3))

for name, cfg in variations.items():
    raw_preds = run_classification(val_data, cfg["prompt"], labels=cfg["labels"])
    mapped_preds = [cfg["label_map"].get(p, "unknown") for p in raw_preds]
    val_data[f'pred_{name}'] = mapped_preds
    print(f"\n{name.upper()}:")
    print(classification_report(val_data['stance_cat'], mapped_preds, digits=3))

# How many tweets got the same label across ALL variants?
pred_cols = ['pred_zero'] + [f'pred_{n}' for n in variations.keys()]
agreement = val_data[pred_cols].apply(lambda row: len(set(row)) == 1, axis=1)
print(f"{agreement.sum()} / {len(val_data)} tweets ({100*agreement.mean():.1f}%) "
      f"received the SAME label across all variants.")
print(f"{(~agreement).sum()} tweets changed classification depending on wording.")

In [ ]:
#@title Solution: Exercise 5 (Chain-of-Thought)
prompt_cot = (
    "Read the following tweet about the Women's March.\n\n"
    "Tweet: {text}\n\n"
    "First, explain in one sentence what the author's position seems to be. "
    "Then state your final answer on a new line.\n\n"
    "Reasoning:"
)

def extract_label_cot(response_text, labels=("support", "oppose")):
    lines = response_text.lower().strip().split("\n")
    for line in reversed(lines):
        for label in labels:
            if label in line:
                return label
    return "unknown"

messages = [prompt_cot.format(text=row['text_cleaned']) for _, row in val_data.iterrows()]
cot_preds = []
for i in tqdm(range(0, len(messages), 4), desc="CoT"):
    responses = classify_batch(instruct_pipe, messages[i:i+4], max_new_tokens=120)
    cot_preds.extend([extract_label_cot(r) for r in responses])
val_data['pred_cot'] = cot_preds
print(classification_report(val_data['stance_cat'], val_data['pred_cot'], digits=3))

In [ ]:
#@title Solution: Extension A (Structured Output)
import json as json_lib

prompt_json = (
    "Classify the following tweet about the Women's March.\n\n"
    "Tweet: {text}\n\n"
    "Respond with ONLY a JSON object in this exact format, no other text:\n"
    '{{"label": "support or oppose", "confidence": "high, medium, or low"}}'
)

subset = val_data.head(30)
messages = [prompt_json.format(text=row['text_cleaned']) for _, row in subset.iterrows()]

json_responses = []
for i in range(0, len(messages), 4):
    batch = messages[i:i+4]
    json_responses.extend(classify_batch(instruct_pipe, batch, max_new_tokens=80))

valid = 0
for i, resp in enumerate(json_responses):
    cleaned = resp.strip().strip('`').strip()
    if cleaned.startswith('json'):
        cleaned = cleaned[4:].strip()
    try:
        parsed = json_lib.loads(cleaned)
        if 'label' in parsed and 'confidence' in parsed:
            valid += 1
            if i < 5:
                print(f"  [OK] {parsed}")
    except json_lib.JSONDecodeError:
        if i < 5:
            print(f"  [FAIL] {resp[:100]}")

print(f"\nValid JSON: {valid} / {len(json_responses)} ({100*valid/len(json_responses):.0f}%)")
print("\nA 0.5B model will partially comply with JSON formatting.")
print("Larger models (GPT-4o, Claude, Llama-70B) achieve near-perfect compliance.")

---
*This notebook is part of the [LLMs for Social Science](https://llmsforsocialscience.net) course.*